# Sentimento por aspecto com Qwen3-1.7B + LoRA

Roteiro completo do projeto, para rodar **no VSCode** (kernel `.venv`).
Funciona em GPU Apple Silicon (Metal/MPS) e em GPU NVIDIA (CUDA): o código detecta a GPU sozinho.

> Dica: abra um terminal no VSCode e rode `python src/gpu_monitor.py` para acompanhar o uso da GPU ao vivo durante o treino.

In [ ]:
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, "src")
print("Pasta do projeto:", os.getcwd())

## 1. GPU instalada e funcionando (print para a entrega)

In [ ]:
!{sys.executable} src/gpu_info.py

## 2. Dados
Gera 2000 avaliações de treino, 200 de validação e 400 de teste. O teste usa **frases que não aparecem no treino**.
Também há 40 avaliações escritas à mão em `data/teste_manual.jsonl`.

In [ ]:
!{sys.executable} src/data.py

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 160)
pd.read_json("data/train.jsonl", lines=True).head(8)

## 3. Baseline e experimentos
O baseline é o Qwen3-1.7B **sem treino** (zero-shot), com o mesmo prompt.
Depois são treinados 7 adaptadores LoRA, cada um mudando **um** parâmetro em relação à referência (`r16_lr2e-4`).

⏱️ Na GPU do Apple M5 leva cerca de 4 a 5 horas. Para um teste rápido, use `--quick`.

In [ ]:
# !{sys.executable} src/experiments.py --quick   # teste rápido
!{sys.executable} src/experiments.py

## 4. Resultados

In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown(open("results/RESULTADOS.md").read()))
for img in ["comparacao.png", "f1_por_aspecto.png", "curvas_loss.png"]:
    display(Image(f"results/{img}"))

In [ ]:
import json, glob
runs = [json.load(open(p)) for p in glob.glob("results/*.json") if "sets" in open(p).read()]
best = max(runs, key=lambda r: r["sets"]["teste"]["pares"]["f1"])
print("Melhor experimento:", best["name"])
display(Image("results/baseline_zero_shot_confusao.png"), Image(f"results/{best['name']}_confusao.png"))

## 5. Avaliação das respostas: baseline × modelo treinado no teste manual

In [ ]:
base = pd.read_csv("results/baseline_zero_shot_manual_predicoes.csv")
ft = pd.read_csv(f"results/{best['name']}_manual_predicoes.csv")
pd.DataFrame({"avaliação": base.text, "esperado": base.esperado,
              "baseline": base.previsto, "treinado": ft.previsto, "treinado_correto": ft.correto})

## 6. Testar o modelo treinado aqui mesmo

In [ ]:
from modeling import load_model, analyze, parse_output
model, tok = load_model(f"outputs/{best['name']}")
for t in ["Chegou rapidinho, mas o carregador parou de funcionar no segundo dia.",
          "Caro, porém a qualidade é excepcional. O vendedor respondeu tudo com paciência."]:
    print(t, "->", parse_output(analyze(model, tok, [t])[0]))

## 7. Frontend
Em um terminal do VSCode:
```bash
python app/app.py --adapter outputs/<melhor_experimento>
```
e abra http://localhost:7860.